# Data Cleaning and Pre-Processing: Sentiment140 Datensatz

In [1]:
from rich import print
from propra_webscience_ws24.constants import (
    TRAIN_DATASET_FILE_PATH,
    TEST_DATASET_FILE_PATH,
    PREPARED_TEST_DATASET_FILE_PATH,
    PREPARED_TRAIN_DATASET_FILE_PATH,
)

In [2]:
%%time

import pandas as pd
from dateutil import parser, tz


pd.options.display.max_colwidth = None

df = pd.read_parquet(TRAIN_DATASET_FILE_PATH)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 2 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   sentiment  1600000 non-null  int64 
 1   text       1600000 non-null  object
dtypes: int64(1), object(1)
memory usage: 24.4+ MB
CPU times: user 2.3 s, sys: 531 ms, total: 2.83 s
Wall time: 1.94 s


### Data Cleaning and Preparation - Trainingsdaten

<img src="Bilder/dslc-data-cleaning-and-preparation.png" width="800" />

#### Text-Bereinigung und Tokenization

Schritte:

- Analog zum Paper werden Nennungen von Nutzern `@<user-name>`, nicht alpha-numerische Worte und URLs entfernt (s. RegEx).
- Verkürzte Wortformen werden mittels `contractions.fix()` aufgelöst.
- Der Text wird in Tokens aufgeteilt (`nltk.word_tokenize`).
- Stop-Worte werden entfernt.
- So erhaltene Token je Tweet werden in neue Spalte `tokenized_text` geschrieben.

In [ ]:
import nltk
from nltk.corpus import stopwords
import re
import contractions

nltk.download('punkt_tab')

REPLACEMENT_PATTERN = re.compile("(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)")
STOP_WORDS = stopwords.words('english')


def clean_text(text: str, remove_stop_words=True) -> list[str]:
    text_mod = contractions.fix(text)
    text_mod = re.sub(REPLACEMENT_PATTERN, " ", text_mod).lower()

    tokens= nltk.word_tokenize(text_mod)
    if not remove_stop_words:
        return tokens
    
    return [w for w in tokens if w not in STOP_WORDS]

CPU times: user 1.64 s, sys: 144 ms, total: 1.79 s
Wall time: 1.68 s


[nltk_data] Downloading package punkt_tab to /home/andi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
%%time

print('Erzeuge Spalte mit Token ohne Stoppworte...')
df['tokenized_text_wo_stop_words'] = df.text.apply(lambda x: clean_text(x, remove_stop_words=True))
print('Erzeuge Spalte mit Token mit Stoppworten...')
df['tokenized_text_w_stop_words'] = df.text.apply(lambda x: clean_text(x, remove_stop_words=False))
print('Spalten mit Token erzeugt.')

#### Stemming/Lemmatization

Ergänzung von vier Spalten, in denen die Token (s. Spalte `tokenized_text`) auf ihre korrekte Grundform (s. `lemmatized_tokens`) bzw. eine heuristische Basisform (s. `stemmed_tokens`) geändert werden jeweils für die Spalte mit/ohne Entfernung der Stoppworte.

In [8]:
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download("wordnet")


lemmatizer = WordNetLemmatizer()


def lemmatization_of_tweet(text):
    return " ".join([lemmatizer.lemmatize(w) for w in text])


stemmer = PorterStemmer()


def stemming_of_tweet(text):
    return " ".join([stemmer.stem(w) for w in text])

[nltk_data] Downloading package wordnet to /home/andi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
%%time

print('Erzeuge Spalte mit Lemmatization Token ohne Stoppworte...')
df['lemmatized_tokens_wo_stop_words'] = df.tokenized_text_wo_stop_words.apply(lambda x: lemmatization_of_tweet(x))
print('Erzeuge Spalte mit Lemmatization Token mit Stoppworten...')
df['lemmatized_tokens_w_stop_words'] = df.tokenized_text_w_stop_words.apply(lambda x: lemmatization_of_tweet(x))
print('Spalten mit Token erzeugt.')
df.head(10)

In [ ]:
%%time

print('Erzeuge Spalte mit Stemming Token ohne Stoppworte...')
df['stemmed_tokens_wo_stop_words'] = df.tokenized_text_wo_stop_words.apply(lambda x: ' '.join([stemmer.stem(w) for w in x]))
print('Erzeuge Spalte mit Stemming Token mit Stoppworten...')
df['stemmed_tokens_w_stop_words'] = df.tokenized_text_w_stop_words.apply(lambda x: ' '.join([stemmer.stem(w) for w in x]))
print('Spalten mit Tokens erzeugt.')
df.head(10)

Erzeuge Spalte mit Stemming Token ohne Stoppworte...

Erzeuge Spalte mit Stemming Token mit Stoppworten...

Spalten mit Tokens erzeugt.

CPU times: user 7min 14s, sys: 348 ms, total: 7min 14s
Wall time: 7min 16s


,sentiment,text,tokenized_text_wo_stop_words,tokenized_text_w_stop_words,lemmatized_tokens_wo_stop_words,lemmatized_tokens_w_stop_words,stemmed_tokens_wo_stop_words,stemmed_tokens_w_stop_words
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D","[awww, bummer, shoulda, got, david, carr, third, day]","[awww, that, is, a, bummer, you, shoulda, got, david, carr, of, third, day, to, do, it, d]",awww bummer shoulda got david carr third day,awww that is a bummer you shoulda got david carr of third day to do it d,awww bummer shoulda got david carr third day,awww that is a bummer you shoulda got david carr of third day to do it d
1,0,is upset that he can't update his Facebook by texting it... and might cry as a result School today also. Blah!,"[upset, update, facebook, texting, might, cry, result, school, today, also, blah]","[is, upset, that, he, can, not, update, his, facebook, by, texting, it, and, might, cry, as, a, result, school, today, also, blah]",upset update facebook texting might cry result school today also blah,is upset that he can not update his facebook by texting it and might cry a a result school today also blah,upset updat facebook text might cri result school today also blah,is upset that he can not updat hi facebook by text it and might cri as a result school today also blah
2,0,@Kenichan I dived many times for the ball. Managed to save 50% The rest go out of bounds,"[dived, many, times, ball, managed, save, 50, rest, go, bounds]","[i, dived, many, times, for, the, ball, managed, to, save, 50, the, rest, go, out, of, bounds]",dived many time ball managed save 50 rest go bound,i dived many time for the ball managed to save 50 the rest go out of bound,dive mani time ball manag save 50 rest go bound,i dive mani time for the ball manag to save 50 the rest go out of bound
3,0,my whole body feels itchy and like its on fire,"[whole, body, feels, itchy, like, fire]","[my, whole, body, feels, itchy, and, like, its, on, fire]",whole body feel itchy like fire,my whole body feel itchy and like it on fire,whole bodi feel itchi like fire,my whole bodi feel itchi and like it on fire
4,0,"@nationwideclass no, it's not behaving at all. i'm mad. why am i here? because I can't see you all over there.","[behaving, mad, see]","[no, it, is, not, behaving, at, all, i, am, mad, why, am, i, here, because, i, can, not, see, you, all, over, there]",behaving mad see,no it is not behaving at all i am mad why am i here because i can not see you all over there,behav mad see,no it is not behav at all i am mad whi am i here becaus i can not see you all over there
5,0,@Kwesidei not the whole crew,"[whole, crew]","[not, the, whole, crew]",whole crew,not the whole crew,whole crew,not the whole crew
6,0,Need a hug,"[need, hug]","[need, a, hug]",need hug,need a hug,need hug,need a hug
7,0,"@LOLTrish hey long time no see! Yes.. Rains a bit ,only a bit LOL , I'm fine thanks , how's you ?","[hey, long, time, see, yes, rains, bit, bit, lol, fine, thanks]","[hey, long, time, no, see, yes, rains, a, bit, only, a, bit, lol, i, am, fine, thanks, how, is, you]",hey long time see yes rain bit bit lol fine thanks,hey long time no see yes rain a bit only a bit lol i am fine thanks how is you,hey long time see ye rain bit bit lol fine thank,hey long time no see ye rain a bit onli a bit lol i am fine thank how is you
8,0,@Tatiana_K nope they didn't have it,"[k, nope]","[k, nope, they, did, not, have, it]",k nope,k nope they did not have it,k nope,k nope they did not have it
9,0,@twittera que me muera ?,"[que, muera]","[que, me, muera]",que muera,que me muera,que muera,que me muera


#### Speicherung des bereinigten Datensatzes

In [6]:
df.to_parquet(PREPARED_TRAIN_DATASET_FILE_PATH, compression='snappy')

### Data Cleaning and Preparation - Testdaten

In [6]:
df_test = pd.read_parquet(TEST_DATASET_FILE_PATH)
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  498 non-null    int64 
 1   query      498 non-null    object
 2   text       498 non-null    object
dtypes: int64(1), object(2)
memory usage: 11.8+ KB


#### Bereinigung

In [9]:
%%time

print('Erzeuge Spalten mit Token, angepasstem und Tweets mit Lemmatization/Stemming...')
print('Erzeuge Spalten mit Token...')
df_test['tokenized_text_wo_stop_words'] = df_test.text.apply(lambda x: clean_text(x, remove_stop_words=True))
df_test['tokenized_text_w_stop_words'] = df_test.text.apply(lambda x: clean_text(x, remove_stop_words=False))

print('Erzeuge Spalten mit Lemmatization...')
df_test['lemmatized_tokens_wo_stop_words'] = df_test.tokenized_text_wo_stop_words.apply(lambda x: lemmatization_of_tweet(x))
df_test['lemmatized_tokens_w_stop_words'] = df_test.tokenized_text_w_stop_words.apply(lambda x: lemmatization_of_tweet(x))

print('Erzeuge Spalten mit Stemming...')
df_test['stemmed_tokens_wo_stop_words'] = df_test.tokenized_text_wo_stop_words.apply(lambda x: ' '.join([stemmer.stem(w) for w in x]))
df_test['stemmed_tokens_w_stop_words'] = df_test.tokenized_text_w_stop_words.apply(lambda x: ' '.join([stemmer.stem(w) for w in x]))
print('Spalten erzeugt.')

Erzeuge Spalten mit Token, angepasstem und Tweets mit Lemmatization/Stemming...

Erzeuge Spalten mit Token...

Erzeuge Spalten mit Lemmatization...

Erzeuge Spalten mit Stemming...

Spalten erzeugt.

CPU times: user 328 ms, sys: 2.05 ms, total: 330 ms
Wall time: 330 ms


#### Speicherung des bereinigten Datensatzes

In [10]:
df_test.to_parquet(PREPARED_TEST_DATASET_FILE_PATH, compression='snappy')